In [1]:
import torch
import numpy as np
import evaluate
import json
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    TrainerCallback
)

# --- 1. GLOBAL INITIALIZATION (Fixes image_3a45d9.png) ---
model_id = "Salesforce/codet5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
rouge_metric = evaluate.load("rouge")

c:\Programming\Python\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(
c:\Programming\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [2]:
# --- 2. FUNCTIONS (Now they can "see" the tokenizer) ---
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}

def preprocess_function(examples):
    inputs = [f"explain {lang}: {code}" for lang, code in zip(examples["language"], examples["input"])]
    model_inputs = tokenizer(inputs, max_length=256, padding="max_length", truncation=True)
    labels = tokenizer(text_target=examples["output"], max_length=128, padding="max_length", truncation=True)
    labels["input_ids"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# --- 3. CUSTOM CALLBACK FOR PROFESSOR'S MILESTONES ---
class MilestoneEvalCallback(TrainerCallback):
    def __init__(self, milestones, batch_size):
        self.milestones = sorted(milestones)
        self.batch_size = batch_size
        self.evaluated_milestones = set()

    def on_step_end(self, args, state, control, **kwargs):
        # Calculate roughly how many samples have been processed
        samples_seen = state.global_step * self.batch_size
        for m in self.milestones:
            if samples_seen >= m and m not in self.evaluated_milestones:
                print(f"\nMILESTONE REACHED: {m} samples. Evaluating now...")
                control.should_evaluate = True
                self.evaluated_milestones.add(m)
                break
        return control

In [ ]:
import json
import re

data = []

with open('../data/dataset.json', 'r', encoding='utf-8') as f:
    # Filter out the structural JSON characters like [ ] { } ,
    # to leave only the lines containing our actual data keys
    lines = [line.strip() for line in f if any(key in line for key in ['"input":', '"output":', '"language":'])]

# We expect 3 lines per sample. We'll iterate in steps of 3.
for i in range(0, len(lines) - 2, 3):
    try:
        def extract_content(line):
            # Find the first colon (separating key from value)
            start_index = line.find(':') + 1
            # Find the first and last double quotes after that colon
            first_quote = line.find('"', start_index) + 1
            last_quote = line.rfind('"')
            return line[first_quote:last_quote]

        # Map the lines to our dictionary keys
        sample = {
            "input": extract_content(lines[i]),
            "output": extract_content(lines[i+1]),
            "language": extract_content(lines[i+2])
        }
        data.append(sample)
    except Exception as e:
        # If a line is malformed, skip it and keep going
        continue

print(f"Successfully got {len(data)} samples!")

# --- SAVE IT PROPERLY IMMEDIATELY ---
# This ensures we have a valid JSON file for the next time
with open('../data/dataset_fixed.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

print("Saved data to 'dataset_fixed.json'")

Successfully got 15597 samples!
Saved data to 'dataset_fixed.json'


In [ ]:
with open('../data/dataset_fixed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Now Python will load the data correctly
hf_dataset = Dataset.from_list(data)
split_dataset = hf_dataset.train_test_split(test_size=0.1)
tokenized_datasets = split_dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 1560/1560 [00:01<00:00, 1373.90 examples/s]


In [ ]:
# --- 5. TRAINING SETUP (Optimized for RTX 3060) ---
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
batch_size = 16 

training_args = Seq2SeqTrainingArguments(
    output_dir="../results/codet5-beginner-explainer",
    eval_strategy="steps",       # <--- FIXED: Changed from 'evaluation_strategy'
    eval_steps=999999,           # Only trigger via our callback milestones
    learning_rate=3e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,                   # Uses RTX 3060 Tensor Cores
    logging_steps=10,
    save_total_limit=1,
    report_to="none"
)

# Milestones as requested: 50, 100, 500, 1000, 5000
sample_milestones = [50, 100, 500, 1000, 5000,10000]

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
    callbacks=[MilestoneEvalCallback(sample_milestones, batch_size)]
)

C:\Users\kushj\AppData\Local\Temp\ipykernel_10476\1407613049.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
# --- 6. START ---
print("Starting fine-tuning with milestone tracking...")
trainer.train()

# --- ADD THIS FOR THE FINAL SCORE ---
print("\n Training Complete. Running FINAL evaluation on the full dataset...")
final_results = trainer.evaluate()
print(f"FINAL ROUGE SCORES: {final_results}")

# Final Save
model.save_pretrained("../results/final_codet5_model")
tokenizer.save_pretrained("../results/final_codet5_model")
print("All done! Check your console logs for the ROUGE % at each milestone and the final score.")

Starting fine-tuning with milestone tracking...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
4,No log,5.266997,10.490000,2.080000,9.460000,9.480000
7,No log,3.668875,13.940000,3.960000,12.480000,12.500000
32,2.989000,2.335217,27.420000,15.660000,24.300000,24.250000
63,2.451500,1.870388,33.290000,20.580000,30.070000,30.090000
313,1.657500,1.477594,45.970000,33.520000,43.410000,43.430000
625,1.747600,1.369579,48.480000,35.590000,45.570000,45.550000



[!] MILESTONE REACHED: 50 samples. Evaluating now...

[!] MILESTONE REACHED: 100 samples. Evaluating now...

[!] MILESTONE REACHED: 500 samples. Evaluating now...

[!] MILESTONE REACHED: 1000 samples. Evaluating now...

[!] MILESTONE REACHED: 5000 samples. Evaluating now...

[!] MILESTONE REACHED: 10000 samples. Evaluating now...

 Training Complete. Running FINAL evaluation on the full dataset...


FINAL ROUGE SCORES: {'eval_loss': 1.2258137464523315, 'eval_rouge1': 51.19, 'eval_rouge2': 38.43, 'eval_rougeL': 48.23, 'eval_rougeLsum': 48.26, 'eval_runtime': 115.9378, 'eval_samples_per_second': 13.455, 'eval_steps_per_second': 0.845, 'epoch': 3.0}
All done! Check your console logs for the ROUGE % at each milestone and the final score.
